# P01 — Estimation du prix au m² des ventes immobilières (DVF)

**Contexte métier.** Agents immobiliers, notaires et plateformes d'estimation (DVF+ de la DGFiP) : estimer instantanément la valeur d'un bien à partir de ses caractéristiques. Enjeu : transparence des marchés, négociation, aide à la décision d'achat.

**Objectif de l'analyse exploratoire.** Identifier la variable cible (le prix au m²) et les régresseurs exploitables (surface, type, localisation, mois de vente), quantifier leur lien avec le prix.

**Données.** Demande de valeurs foncières (DVF) 2024 — DGFiP, data.gouv.fr. Licence ouverte Etalab 2.0. Fichier annuel de 3,5 millions de mutations (~450 Mo décompressés).

## 1. Téléchargement

Archive DVF 2024 sur data.gouv.fr (65 Mo compressés, ~450 Mo décompressés) :

In [ ]:
import urllib.request, zipfile, os

URL = "https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20261012-171636/valeursfoncieres-2024.txt.zip"
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/ValeursFoncieres-2024.txt"):
    urllib.request.urlretrieve(URL, "data/valeursfoncieres-2024.zip")
    with zipfile.ZipFile("data/valeursfoncieres-2024.zip") as z:
        z.extractall("data")
print(os.path.getsize("data/ValeursFoncieres-2024.txt") / 1e6, "Mo")

## 2. Chargement (filtrage maison/appartement, échantillon 30 %)

Le fichier complet dépasse 450 Mo : lecture par chunks avec `pandas`, filtrage des ventes de maisons et appartements avec prix et surface valides, échantillonnage à 30 % pour l'analyse.

In [ ]:
import pandas as pd, numpy as np

keep = ["Valeur fonciere", "Code departement", "Code commune", "Type local",
        "Surface reelle bati", "Nombre pieces principales", "Surface terrain", "Date mutation"]
rows = []
for chunk in pd.read_csv("data/ValeursFoncieres-2024.txt", sep="|", usecols=keep,
                         chunksize=300_000, low_memory=False, encoding="utf-8"):
    c = chunk[chunk["Type local"].isin(["Maison", "Appartement"])]
    c = c[pd.to_numeric(c["Valeur fonciere"], errors="coerce").notna()]
    rows.append(c.sample(frac=0.30, random_state=42))
df = pd.concat(rows)
df["prix"] = pd.to_numeric(df["Valeur fonciere"], errors="coerce")
df["surface"] = pd.to_numeric(df["Surface reelle bati"], errors="coerce")
df = df.dropna(subset=["prix", "surface"])
df = df[(df["prix"] > 1000) & (df["surface"] > 9)]
df["prix_m2"] = df["prix"] / df["surface"]
df["mois"] = df["Date mutation"].str[3:5]
print(len(df), "ventes maison/appartement (échantillon 30 %)")

## 3. Analyse

Distribution du prix au m², variations par type, département, surface, mois — puis corrélations de Spearman (le prix au m² est très asymétrique).

In [ ]:
print(df["prix_m2"].describe(percentiles=[.05, .25, .5, .75, .95]).round(1))
print()
print("Médiane par type :", df.groupby("Type local")["prix_m2"].median().round(0).to_dict())
print("Top départements :", df.groupby("Code departement")["prix_m2"].median().round(0).sort_values(ascending=False).head(5).to_dict())
print("Médiane par décile de surface :")
q = pd.qcut(df["surface"], 10, duplicates="drop")
print(df.groupby(q, observed=True)["prix_m2"].median().round(0).to_dict())
num = df[["prix_m2", "surface", "Nombre pieces principales", "Surface terrain"]].apply(pd.to_numeric, errors="coerce")
print("Corrélations Spearman avec prix_m2 :")
print(num.corr("spearman")["prix_m2"].round(2))

## 4. Résultats observés (exécution du 16/10/2026)

- n lignes brutes : 3505340
- n ventes maisons apparts : 995131
- n echantillon eda : 282814
- prix m2 stats : count : 282 814.00, mean : 3 616.60, std : 3 540.00, min : 50.40, 5% : 728.60, 25% : 1 653.40, 50% : 2 585.20, 75% : …
- prix m2 median par type local : Appartement : 3 476.00, Maison : 2 136.00
- prix m2 median par dept top10 : 75 : 9 709.00, 92 : 6 625.00, 94 : 5 126.00, 06 : 4 955.00, 74 : 4 500.00, 2A : 4 283.00, 83 : 4 194.00, 93 : 4 174.0…
- prix m2 median par dept bottom5 : 23 : 912.00, 52 : 1 000.00, 55 : 1 118.00, 36 : 1 119.00, 58 : 1 137.00
- mediane par nb pieces : 0.0 : 3 736.00, 1.0 : 4 375.00, 2.0 : 3 400.00, 3.0 : 2 588.00, 4.0 : 2 308.00, 5.0 : 2 174.00, 6.0 : 2 079.00, 7.0 :…
- mediane m2 par surface decile : (8.999, 32.0] : 5 000.00, (32.0, 45.0] : 3 743.00, (45.0, 57.0] : 2 946.00, (57.0, 67.0] : 2 619.00, (67.0, 76.0] : 2…
- corr spearman prix m2 : prix_m2 : 1.00, Surface reelle bati : -0.33, Nombre pieces principales : -0.29, Surface terrain : 0.01

## 5. Variable cible et régresseurs

**Cible (régression) : `prix_m2` = `Valeur fonciere` / `Surface reelle bati`.** On prédit le prix au m² plutôt que le prix total : c'est le standard des estimations et cela rend le modèle généralisable entre surfaces.

**Régresseurs retenus :**
- `Surface reelle bati` (corr. Spearman −0,33) et `Nombre pieces principales` (−0,29) : le prix au m² diminue quand le bien grandit
- `Type local` (maison/appartement), `Surface terrain`
- `Code departement` / `Code commune` : la localisation est le premier facteur de variabilité (Paris 75 : 9 709 €/m² contre 912 €/m² dans la Creuse)
- `mois` (saisonnalité faible mais réelle)

**Pièges :**
- Une ligne DVF peut contenir plusieurs lots (ventes en bloc) : filtrer prix>1000 € et surface>9 m², sinon prix/m² absurdes (médiane observée 2 585 €/m², mais min 50 € et max 30 000 €/m²)
- Distribution très asymétrique : travailler en log-prix, évaluer en MAE sur le log
- 450 Mo de données : chunking obligatoire, échantillon suffisant pour l'EDA
- Les ventes de terrain/activité sont exclues volontairement pour homogénéiser la cible